# Document Pipeline — Independent ETL Walkthrough

A self-contained notebook using entirely invented records. It is not connected to a private database.

This illustrates relational-style source records → project documents → stable content hashes.


In [ ]:
from hashlib import sha256
import json

projects = [{"id": "sample-101", "name": "Cedar Residence", "estimated_material_cents": 1450000}]
expenses = [{"id": "cost-1", "project_id": "sample-101", "amount_cents": 42000, "description": "Site transport"}]
projects, expenses


In [ ]:
documents = []
for project in projects:
    related = [row for row in expenses if row["project_id"] == project["id"]]
    payload = {"project": project, "financial": related}
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    documents.append({"doc_id": "doc-" + project["id"], "payload": payload, "lineage": [row["id"] for row in related], "checksum": sha256(canonical.encode()).hexdigest()})
documents


## Idempotency

The checksum is stable when the input record structure has not changed. Recomputing the same document therefore does not require a new version. A real pipeline would also handle out-of-order events, tenant boundaries, and late corrections.


In [ ]:
first = documents[0]["checksum"]
again = sha256(json.dumps(documents[0]["payload"], sort_keys=True, separators=(",", ":")).encode()).hexdigest()
assert first == again
print("Stable content hash verified:", first[:12])
